# Indonesian Toxicity Detection - Kaggle Training Notebook

This notebook trains all three models in the BEAM architecture for Indonesian toxicity detection, optimized for Kaggle environment.

## 🎯 Models to Train
1. **Tier 1 (Basic)**: TF-IDF + Logistic Regression (~1-5ms latency)
2. **Tier 2 (Contextual)**: BiLSTM (~10-50ms latency)  
3. **Tier 3 (Sociolinguistic)**: IndoBERT (~50-200ms latency)

## 🚀 Kaggle Setup
- Enable GPU T4 x2 for faster training
- Enable internet access for IndoBERT model download
- Expected runtime: TF-IDF (~2min), BiLSTM (~5-10min), IndoBERT (~15-30min)


In [18]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [1]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install transformers datasets accelerate
!pip install scikit-learn pandas numpy matplotlib seaborn
!pip install loguru rich typer
!pip install sastrawi nltk
!pip install plotly

print("✅ All dependencies installed successfully!")


Looking in indexes: https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 51.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 37.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 53.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.9/663.9 MB 6.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 9.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 20.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 MB 36.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.2/128.2 MB 24.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.1/204.1 MB 17.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 MB 22.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
# Import all required libraries
import sys
import os
import time
import warnings
import pickle
import re
import gc
from pathlib import Path
from typing import Any, Dict, List, Optional, Union, Tuple
from abc import ABC, abstractmethod

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW

# Suppress warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully!")
print(f"🐍 Python version: {sys.version}")
print(f"🔥 PyTorch version: {torch.__version__}")
print(f"🚀 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")


✅ All libraries imported successfully!
🐍 Python version: 3.12.3 (main, Aug 14 2025, 17:47:21) [GCC 13.3.0]
🔥 PyTorch version: 2.7.1+cu118
🚀 CUDA available: True
🎮 GPU: Quadro RTX 8000


## Core Classes Implementation


In [5]:
# Base Model Interface
class BaseModel(ABC):
    """Abstract base class for all toxicity detection models."""
    
    def __init__(self, name: str = "base_model") -> None:
        self.name = name
        self.is_trained = False
        self.training_time: Optional[float] = None
        self._model: Optional[Any] = None
        
    @abstractmethod
    def train(self, X_train: List[str], y_train: np.ndarray, 
              X_val: Optional[List[str]] = None, y_val: Optional[np.ndarray] = None) -> Dict[str, Any]:
        pass
    
    @abstractmethod
    def predict(self, texts: List[str]) -> np.ndarray:
        pass
    
    @abstractmethod
    def predict_proba(self, texts: List[str]) -> np.ndarray:
        pass
    
    @abstractmethod
    def save(self, path: Union[str, Path]) -> None:
        pass
    
    @abstractmethod
    def load(self, path: Union[str, Path]) -> None:
        pass
    
    def _validate_trained(self) -> None:
        if not self.is_trained:
            raise ValueError(f"Model '{self.name}' is not trained. Call train() first.")
    
    def _measure_training_time(self, start_time: float) -> float:
        self.training_time = time.perf_counter() - start_time
        return self.training_time

# Indonesian Text Preprocessor
class IndonesianTextPreprocessor:
    def __init__(self, lowercase: bool = True, remove_urls: bool = True, 
                 remove_mentions: bool = True, remove_numbers: bool = False,
                 remove_punctuation: bool = False, use_stemming: bool = False, 
                 min_length: int = 2) -> None:
        self.lowercase = lowercase
        self.remove_urls = remove_urls
        self.remove_mentions = remove_mentions
        self.remove_numbers = remove_numbers
        self.remove_punctuation = remove_punctuation
        self.use_stemming = use_stemming
        self.min_length = min_length
        
        self.stopwords = {
            "yang", "dan", "di", "ke", "dari", "untuk", "pada", "dengan", "adalah",
            "ini", "itu", "atau", "dalam", "juga", "akan", "ada", "oleh", "tidak",
            "si", "saya", "anda", "kamu", "dia", "kami", "kita", "mereka",
            "ya", "sudah", "bisa", "mau", "harus", "lebih", "sangat", "saja",
        }
    
    def clean_text(self, text: str) -> str:
        if self.remove_urls:
            text = re.sub(r"http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+", "", text)
            text = re.sub(r"www\.[a-zA-Z0-9-]+\.[a-zA-Z]{2,}", "", text)
        
        if self.remove_mentions:
            text = re.sub(r"@\w+", "", text)
        
        if self.remove_numbers:
            text = re.sub(r"\d+", "", text)
        
        text = re.sub(r"\s+", " ", text)
        text = text.strip()
        return text
    
    def preprocess(self, text: str, remove_stopwords: bool = False) -> str:
        text = self.clean_text(text)
        
        if self.lowercase:
            text = text.lower()
        
        if self.remove_punctuation:
            text = re.sub(r"[^\w\s]", "", text)
        
        tokens = text.split()
        tokens = [t for t in tokens if len(t) >= self.min_length]
        
        if remove_stopwords:
            tokens = [t for t in tokens if t not in self.stopwords]
        
        return " ".join(tokens)
    
    def preprocess_batch(self, texts: List[str], remove_stopwords: bool = False) -> List[str]:
        return [self.preprocess(text, remove_stopwords) for text in texts]

print("✅ Core classes defined!")


✅ Core classes defined!


In [6]:
# TF-IDF Model Implementation
class TFIDFModel(BaseModel):
    """TF-IDF + Logistic Regression model for basic toxicity detection."""
    
    def __init__(self, name: str = "tfidf_lr", max_features: int = 10000,
                 ngram_range: tuple = (1, 2), min_df: int = 2, max_df: float = 0.95,
                 C: float = 1.0, random_state: int = 42, use_preprocessing: bool = True) -> None:
        super().__init__(name)
        
        self.max_features = max_features
        self.ngram_range = ngram_range
        self.min_df = min_df
        self.max_df = max_df
        self.C = C
        self.random_state = random_state
        self.use_preprocessing = use_preprocessing
        
        self.preprocessor = IndonesianTextPreprocessor(
            lowercase=True, remove_urls=True, remove_mentions=True,
            remove_punctuation=False, use_stemming=False
        ) if use_preprocessing else None
        
        self.vectorizer = TfidfVectorizer(
            max_features=max_features, ngram_range=ngram_range,
            min_df=min_df, max_df=max_df, lowercase=True, stop_words=None
        )
        
        self.classifier = LogisticRegression(
            C=C, random_state=random_state, max_iter=1000, class_weight="balanced"
        )
        
        self.calibrated_classifier = CalibratedClassifierCV(
            self.classifier, method="isotonic", cv="prefit"
        )
        
        self.pipeline = Pipeline([
            ("vectorizer", self.vectorizer),
            ("classifier", self.calibrated_classifier),
        ])
    
    def _preprocess_texts(self, texts: List[str]) -> List[str]:
        if not self.use_preprocessing or self.preprocessor is None:
            return texts
        return self.preprocessor.preprocess_batch(texts, remove_stopwords=True)
    
    def train(self, X_train: List[str], y_train: np.ndarray,
              X_val: Optional[List[str]] = None, y_val: Optional[np.ndarray] = None) -> Dict[str, Any]:
        print(f"🚀 Training {self.name.upper()} model...")
        start_time = time.perf_counter()
        
        X_train_processed = self._preprocess_texts(X_train)
        
        # First fit the vectorizer and transform the data
        X_train_vec = self.vectorizer.fit_transform(X_train_processed)
        
        # Fit the base classifier
        self.classifier.fit(X_train_vec, y_train)
        
        # Now fit the calibrated classifier with validation data
        if X_val is not None and y_val is not None:
            X_val_processed = self._preprocess_texts(X_val)
            X_val_vec = self.vectorizer.transform(X_val_processed)
            self.calibrated_classifier.fit(X_val_vec, y_val)
        else:
            # If no validation data, just use the base classifier
            self.calibrated_classifier = self.classifier
        
        # Create the pipeline for predictions
        self.pipeline = Pipeline([
            ("vectorizer", self.vectorizer),
            ("classifier", self.calibrated_classifier),
        ])
        
        y_train_pred = self.pipeline.predict(X_train_processed)
        train_accuracy = accuracy_score(y_train, y_train_pred)
        train_f1 = f1_score(y_train, y_train_pred, average="weighted")
        
        metrics = {
            "train_accuracy": train_accuracy,
            "train_f1": train_f1,
            "training_time": self._measure_training_time(start_time),
            "n_features": len(self.vectorizer.vocabulary_),
        }
        
        if X_val is not None and y_val is not None:
            X_val_processed = self._preprocess_texts(X_val)
            y_val_pred = self.pipeline.predict(X_val_processed)
            val_accuracy = accuracy_score(y_val, y_val_pred)
            val_f1 = f1_score(y_val, y_val_pred, average="weighted")
            metrics.update({"val_accuracy": val_accuracy, "val_f1": val_f1})
        
        self.is_trained = True
        print(f"✅ {self.name.upper()} training completed! F1: {train_f1:.4f}")
        return metrics
    
    def predict(self, texts: List[str]) -> np.ndarray:
        self._validate_trained()
        texts_processed = self._preprocess_texts(texts)
        return self.pipeline.predict(texts_processed)
    
    def predict_proba(self, texts: List[str]) -> np.ndarray:
        self._validate_trained()
        texts_processed = self._preprocess_texts(texts)
        return self.pipeline.predict_proba(texts_processed)[:, 1]
    
    def get_feature_importance(self, top_k: int = 20) -> Dict[str, float]:
        self._validate_trained()
        feature_names = self.vectorizer.get_feature_names_out()
        
        # Always use the base classifier directly since we know it's fitted
        coefficients = self.classifier.coef_[0]
        top_indices = np.argsort(np.abs(coefficients))[-top_k:][::-1]
        return {feature_names[i]: float(coefficients[i]) for i in top_indices}
    
    def save(self, path: Union[str, Path]) -> None:
        self._validate_trained()
        path = Path(path)
        path.mkdir(parents=True, exist_ok=True)
        
        with open(path / "model.pkl", "wb") as f:
            pickle.dump(self.pipeline, f)
        
        metadata = {
            "name": self.name, "is_trained": self.is_trained,
            "training_time": self.training_time, "max_features": self.max_features,
            "ngram_range": self.ngram_range, "min_df": self.min_df,
            "max_df": self.max_df, "C": self.C, "random_state": self.random_state,
            "use_preprocessing": self.use_preprocessing
        }
        
        with open(path / "metadata.pkl", "wb") as f:
            pickle.dump(metadata, f)
    
    def load(self, path: Union[str, Path]) -> None:
        path = Path(path)
        if not path.exists():
            raise FileNotFoundError(f"Model directory not found: {path}")
        
        with open(path / "model.pkl", "rb") as f:
            self.pipeline = pickle.load(f)
        
        self.vectorizer = self.pipeline.named_steps["vectorizer"]
        self.calibrated_classifier = self.pipeline.named_steps["classifier"]
        self.classifier = self.calibrated_classifier.base_estimator
        self.is_trained = True

print("✅ TF-IDF Model defined!")


✅ TF-IDF Model defined!


In [7]:
# BiLSTM Model Implementation
class BiLSTMModel(BaseModel):
    """Bidirectional LSTM model for contextual toxicity detection."""
    
    def __init__(self, name: str = "bilstm", vocab_size: int = 10000, embedding_dim: int = 128,
                 hidden_dim: int = 256, num_layers: int = 2, dropout: float = 0.3,
                 max_length: int = 200, epochs: int = 20, batch_size: int = 32,
                 learning_rate: float = 0.001, random_state: int = 42, use_preprocessing: bool = True) -> None:
        super().__init__(name)
        
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.dropout = dropout
        self.max_length = max_length
        self.epochs = epochs
        self.batch_size = batch_size
        self.learning_rate = learning_rate
        self.random_state = random_state
        self.use_preprocessing = use_preprocessing
        
        self.preprocessor = IndonesianTextPreprocessor(
            lowercase=True, remove_urls=True, remove_mentions=True,
            remove_punctuation=False, use_stemming=False
        ) if use_preprocessing else None
        
        self.tokenizer = None
        self.model = None
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    def _preprocess_texts(self, texts: List[str]) -> List[str]:
        if not self.use_preprocessing or self.preprocessor is None:
            return texts
        return self.preprocessor.preprocess_batch(texts, remove_stopwords=True)
    
    def _create_tokenizer(self, texts: List[str]) -> None:
        all_words = []
        for text in texts:
            words = text.lower().split()
            all_words.extend(words)
        
        word_counts = {}
        for word in all_words:
            word_counts[word] = word_counts.get(word, 0) + 1
        
        sorted_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)
        vocab_words = [word for word, _ in sorted_words[:self.vocab_size - 2]]
        
        self.word_to_idx = {
            "<PAD>": 0, "<UNK>": 1,
            **{word: idx + 2 for idx, word in enumerate(vocab_words)}
        }
        self.idx_to_word = {idx: word for word, idx in self.word_to_idx.items()}
    
    def _text_to_sequence(self, texts: List[str]) -> np.ndarray:
        sequences = []
        for text in texts:
            words = text.lower().split()
            sequence = []
            for word in words[:self.max_length]:
                idx = self.word_to_idx.get(word, self.word_to_idx["<UNK>"])
                sequence.append(idx)
            while len(sequence) < self.max_length:
                sequence.append(self.word_to_idx["<PAD>"])
            sequences.append(sequence)
        return np.array(sequences)
    
    def _create_model(self) -> nn.Module:
        class BiLSTMClassifier(nn.Module):
            def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers, dropout):
                super().__init__()
                self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
                self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers, batch_first=True,
                                  dropout=dropout if num_layers > 1 else 0, bidirectional=True)
                self.dropout = nn.Dropout(dropout)
                self.fc = nn.Linear(hidden_dim * 2, 1)
                
            def forward(self, x):
                embedded = self.embedding(x)
                lstm_out, _ = self.lstm(embedded)
                last_output = lstm_out[:, -1, :]
                dropped = self.dropout(last_output)
                output = torch.sigmoid(self.fc(dropped))
                return output.squeeze()
        
        return BiLSTMClassifier(
            vocab_size=len(self.word_to_idx), embedding_dim=self.embedding_dim,
            hidden_dim=self.hidden_dim, num_layers=self.num_layers, dropout=self.dropout
        )
    
    def train(self, X_train: List[str], y_train: np.ndarray,
              X_val: Optional[List[str]] = None, y_val: Optional[np.ndarray] = None) -> Dict[str, Any]:
        print(f"🚀 Training {self.name.upper()} model...")
        start_time = time.perf_counter()
        
        torch.manual_seed(self.random_state)
        np.random.seed(self.random_state)
        
        X_train_processed = self._preprocess_texts(X_train)
        self._create_tokenizer(X_train_processed)
        X_train_seq = self._text_to_sequence(X_train_processed)
        
        self.model = self._create_model().to(self.device)
        
        train_dataset = TensorDataset(torch.LongTensor(X_train_seq), torch.FloatTensor(y_train))
        train_loader = DataLoader(train_dataset, batch_size=self.batch_size, shuffle=True)
        
        criterion = nn.BCELoss()
        optimizer = torch.optim.Adam(self.model.parameters(), lr=self.learning_rate)
        
        self.model.train()
        train_losses = []
        
        for epoch in range(self.epochs):
            epoch_loss = 0.0
            for batch_X, batch_y in train_loader:
                batch_X, batch_y = batch_X.to(self.device), batch_y.to(self.device)
                optimizer.zero_grad()
                outputs = self.model(batch_X)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()
            
            avg_loss = epoch_loss / len(train_loader)
            train_losses.append(avg_loss)
            
            if epoch % 5 == 0:
                print(f"Epoch {epoch+1}/{self.epochs}, Loss: {avg_loss:.4f}")
        
        self.model.eval()
        with torch.no_grad():
            train_pred = self.model(torch.LongTensor(X_train_seq).to(self.device)).cpu().numpy()
            train_pred_binary = (train_pred > 0.5).astype(int)
            train_accuracy = accuracy_score(y_train, train_pred_binary)
            train_f1 = f1_score(y_train, train_pred_binary, average="weighted")
        
        metrics = {
            "train_accuracy": train_accuracy, "train_f1": train_f1,
            "training_time": self._measure_training_time(start_time), "final_loss": train_losses[-1]
        }
        
        if X_val is not None and y_val is not None:
            X_val_processed = self._preprocess_texts(X_val)
            X_val_seq = self._text_to_sequence(X_val_processed)
            with torch.no_grad():
                val_pred = self.model(torch.LongTensor(X_val_seq).to(self.device)).cpu().numpy()
                val_pred_binary = (val_pred > 0.5).astype(int)
                val_accuracy = accuracy_score(y_val, val_pred_binary)
                val_f1 = f1_score(y_val, val_pred_binary, average="weighted")
            metrics.update({"val_accuracy": val_accuracy, "val_f1": val_f1})
        
        self.is_trained = True
        print(f"✅ {self.name.upper()} training completed! F1: {train_f1:.4f}")
        return metrics
    
    def predict(self, texts: List[str]) -> np.ndarray:
        self._validate_trained()
        texts_processed = self._preprocess_texts(texts)
        sequences = self._text_to_sequence(texts_processed)
        self.model.eval()
        with torch.no_grad():
            predictions = self.model(torch.LongTensor(sequences).to(self.device)).cpu().numpy()
        return (predictions > 0.5).astype(int)
    
    def predict_proba(self, texts: List[str]) -> np.ndarray:
        self._validate_trained()
        texts_processed = self._preprocess_texts(texts)
        sequences = self._text_to_sequence(texts_processed)
        self.model.eval()
        with torch.no_grad():
            predictions = self.model(torch.LongTensor(sequences).to(self.device)).cpu().numpy()
        return predictions.astype(np.float64)
    
    def save(self, path: Union[str, Path]) -> None:
        self._validate_trained()
        path = Path(path)
        path.mkdir(parents=True, exist_ok=True)
        
        torch.save(self.model.state_dict(), path / "model.pth")
        
        with open(path / "tokenizer.pkl", "wb") as f:
            pickle.dump({"word_to_idx": self.word_to_idx, "idx_to_word": self.idx_to_word}, f)
        
        metadata = {
            "name": self.name, "is_trained": self.is_trained, "training_time": self.training_time,
            "vocab_size": self.vocab_size, "embedding_dim": self.embedding_dim,
            "hidden_dim": self.hidden_dim, "num_layers": self.num_layers, "dropout": self.dropout,
            "max_length": self.max_length, "epochs": self.epochs, "batch_size": self.batch_size,
            "learning_rate": self.learning_rate, "random_state": self.random_state,
            "use_preprocessing": self.use_preprocessing
        }
        
        with open(path / "metadata.pkl", "wb") as f:
            pickle.dump(metadata, f)
    
    def load(self, path: Union[str, Path]) -> None:
        path = Path(path)
        if not path.exists():
            raise FileNotFoundError(f"Model directory not found: {path}")
        
        with open(path / "tokenizer.pkl", "rb") as f:
            tokenizer_data = pickle.load(f)
            self.word_to_idx = tokenizer_data["word_to_idx"]
            self.idx_to_word = tokenizer_data["idx_to_word"]
        
        self.model = self._create_model().to(self.device)
        self.model.load_state_dict(torch.load(path / "model.pth", map_location=self.device))
        self.is_trained = True

print("✅ BiLSTM Model defined!")


✅ BiLSTM Model defined!


In [8]:
# Transformer Model Implementation
class TransformerModel(BaseModel):
    """IndoBERT transformer model for sociolinguistic toxicity detection."""
    
    def __init__(self, name: str = "indobert", model_name: str = "indobenchmark/indobert-base-p1",
                 max_length: int = 256, epochs: int = 3, batch_size: int = 16,
                 learning_rate: float = 2e-5, warmup_steps: int = 100,
                 random_state: int = 42, use_preprocessing: bool = False) -> None:
        super().__init__(name)
        
        self.model_name = model_name
        self.max_length = max_length
        self.epochs = epochs
        self.batch_size = batch_size
        self.learning_rate = learning_rate
        self.warmup_steps = warmup_steps
        self.random_state = random_state
        self.use_preprocessing = use_preprocessing
        
        self.preprocessor = IndonesianTextPreprocessor(
            lowercase=False, remove_urls=True, remove_mentions=False,
            remove_punctuation=False, use_stemming=False
        ) if use_preprocessing else None
        
        self.tokenizer = None
        self.model = None
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    def _preprocess_texts(self, texts: List[str]) -> List[str]:
        if not self.use_preprocessing or self.preprocessor is None:
            return texts
        return self.preprocessor.preprocess_batch(texts, remove_stopwords=False)
    
    def _initialize_tokenizer_and_model(self) -> None:
        print(f"Loading IndoBERT tokenizer and model: {self.model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        
        class IndoBERTClassifier(nn.Module):
            def __init__(self, model_name):
                super().__init__()
                self.bert = AutoModel.from_pretrained(model_name)
                self.dropout = nn.Dropout(0.1)
                self.classifier = nn.Linear(self.bert.config.hidden_size, 1)
                
            def forward(self, input_ids, attention_mask):
                outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
                pooled_output = outputs.pooler_output
                dropped = self.dropout(pooled_output)
                logits = torch.sigmoid(self.classifier(dropped))
                return logits.squeeze()
        
        self.model = IndoBERTClassifier(self.model_name).to(self.device)
    
    def _tokenize_texts(self, texts: List[str]) -> Dict[str, torch.Tensor]:
        return self.tokenizer(texts, truncation=True, padding=True, 
                            max_length=self.max_length, return_tensors="pt")
    
    def train(self, X_train: List[str], y_train: np.ndarray,
              X_val: Optional[List[str]] = None, y_val: Optional[np.ndarray] = None) -> Dict[str, Any]:
        print(f"🚀 Training {self.name.upper()} model...")
        start_time = time.perf_counter()
        
        torch.manual_seed(self.random_state)
        np.random.seed(self.random_state)
        
        X_train_processed = self._preprocess_texts(X_train)
        self._initialize_tokenizer_and_model()
        
        train_encodings = self._tokenize_texts(X_train_processed)
        
        train_dataset = TensorDataset(
            train_encodings["input_ids"], train_encodings["attention_mask"], torch.FloatTensor(y_train)
        )
        train_loader = DataLoader(train_dataset, batch_size=self.batch_size, shuffle=True)
        
        criterion = nn.BCELoss()
        optimizer = AdamW(self.model.parameters(), lr=self.learning_rate)
        
        total_steps = len(train_loader) * self.epochs
        scheduler = get_linear_schedule_with_warmup(
            optimizer, num_warmup_steps=self.warmup_steps, num_training_steps=total_steps
        )
        
        self.model.train()
        train_losses = []
        
        for epoch in range(self.epochs):
            epoch_loss = 0.0
            for batch_input_ids, batch_attention_mask, batch_y in train_loader:
                batch_input_ids = batch_input_ids.to(self.device)
                batch_attention_mask = batch_attention_mask.to(self.device)
                batch_y = batch_y.to(self.device)
                
                optimizer.zero_grad()
                outputs = self.model(batch_input_ids, batch_attention_mask)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
                scheduler.step()
                
                epoch_loss += loss.item()
            
            avg_loss = epoch_loss / len(train_loader)
            train_losses.append(avg_loss)
            print(f"Epoch {epoch+1}/{self.epochs}, Loss: {avg_loss:.4f}")
        
        self.model.eval()
        with torch.no_grad():
            train_pred = self.model(train_encodings["input_ids"].to(self.device), 
                                  train_encodings["attention_mask"].to(self.device)).cpu().numpy()
            train_pred_binary = (train_pred > 0.5).astype(int)
            train_accuracy = accuracy_score(y_train, train_pred_binary)
            train_f1 = f1_score(y_train, train_pred_binary, average="weighted")
        
        metrics = {
            "train_accuracy": train_accuracy, "train_f1": train_f1,
            "training_time": self._measure_training_time(start_time), "final_loss": train_losses[-1]
        }
        
        if X_val is not None and y_val is not None:
            X_val_processed = self._preprocess_texts(X_val)
            val_encodings = self._tokenize_texts(X_val_processed)
            with torch.no_grad():
                val_pred = self.model(val_encodings["input_ids"].to(self.device),
                                     val_encodings["attention_mask"].to(self.device)).cpu().numpy()
                val_pred_binary = (val_pred > 0.5).astype(int)
                val_accuracy = accuracy_score(y_val, val_pred_binary)
                val_f1 = f1_score(y_val, val_pred_binary, average="weighted")
            metrics.update({"val_accuracy": val_accuracy, "val_f1": val_f1})
        
        self.is_trained = True
        print(f"✅ {self.name.upper()} training completed! F1: {train_f1:.4f}")
        return metrics
    
    def predict(self, texts: List[str]) -> np.ndarray:
        self._validate_trained()
        texts_processed = self._preprocess_texts(texts)
        encodings = self._tokenize_texts(texts_processed)
        self.model.eval()
        with torch.no_grad():
            predictions = self.model(encodings["input_ids"].to(self.device),
                                   encodings["attention_mask"].to(self.device)).cpu().numpy()
        return (predictions > 0.5).astype(int)
    
    def predict_proba(self, texts: List[str]) -> np.ndarray:
        self._validate_trained()
        texts_processed = self._preprocess_texts(texts)
        encodings = self._tokenize_texts(texts_processed)
        self.model.eval()
        with torch.no_grad():
            predictions = self.model(encodings["input_ids"].to(self.device),
                                   encodings["attention_mask"].to(self.device)).cpu().numpy()
        return predictions.astype(np.float64)
    
    def get_attention_weights(self, text: str) -> Tuple[List[str], np.ndarray]:
        self._validate_trained()
        encoding = self.tokenizer(text, truncation=True, padding=True, 
                                max_length=self.max_length, return_tensors="pt")
        tokens = self.tokenizer.convert_ids_to_tokens(encoding["input_ids"][0])
        
        self.model.eval()
        with torch.no_grad():
            outputs = self.model.bert(encoding["input_ids"].to(self.device),
                                    encoding["attention_mask"].to(self.device),
                                    output_attentions=True)
            attention = outputs.attentions[-1][0].cpu().numpy()
            attention_avg = attention.mean(axis=0)
        
        return tokens, attention_avg
    
    def save(self, path: Union[str, Path]) -> None:
        self._validate_trained()
        path = Path(path)
        path.mkdir(parents=True, exist_ok=True)
        
        torch.save(self.model.state_dict(), path / "model.pth")
        self.tokenizer.save_pretrained(path / "tokenizer")
        
        metadata = {
            "name": self.name, "is_trained": self.is_trained, "training_time": self.training_time,
            "model_name": self.model_name, "max_length": self.max_length, "epochs": self.epochs,
            "batch_size": self.batch_size, "learning_rate": self.learning_rate,
            "warmup_steps": self.warmup_steps, "random_state": self.random_state,
            "use_preprocessing": self.use_preprocessing
        }
        
        with open(path / "metadata.pkl", "wb") as f:
            pickle.dump(metadata, f)
    
    def load(self, path: Union[str, Path]) -> None:
        path = Path(path)
        if not path.exists():
            raise FileNotFoundError(f"Model directory not found: {path}")
        
        self.tokenizer = AutoTokenizer.from_pretrained(path / "tokenizer")
        self._initialize_tokenizer_and_model()
        self.model.load_state_dict(torch.load(path / "model.pth", map_location=self.device))
        self.is_trained = True

print("✅ Transformer Model defined!")


✅ Transformer Model defined!


## Evaluation Metrics Implementation


In [9]:
# Evaluation Metrics Implementation
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
from sklearn.calibration import calibration_curve

def compute_metrics(y_true, y_pred, y_proba, latencies=None):
    """Compute comprehensive evaluation metrics."""
    accuracy = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="weighted")
    roc_auc = roc_auc_score(y_true, y_proba)
    pr_auc = average_precision_score(y_true, y_proba)
    
    # Expected Calibration Error (ECE)
    ece = compute_ece(y_true, y_proba)
    
    # Average latency
    avg_latency_ms = np.mean(latencies) * 1000 if latencies else None
    
    return {
        "accuracy": accuracy,
        "f1": f1,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "ece": ece,
        "avg_latency_ms": avg_latency_ms
    }

def compute_ece(y_true, y_proba, n_bins=10):
    """Compute Expected Calibration Error."""
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    bin_lowers = bin_boundaries[:-1]
    bin_uppers = bin_boundaries[1:]
    
    ece = 0
    for bin_lower, bin_upper in zip(bin_lowers, bin_uppers):
        in_bin = (y_proba > bin_lower) & (y_proba <= bin_upper)
        prop_in_bin = in_bin.mean()
        
        if prop_in_bin > 0:
            accuracy_in_bin = y_true[in_bin].mean()
            avg_confidence_in_bin = y_proba[in_bin].mean()
            ece += np.abs(avg_confidence_in_bin - accuracy_in_bin) * prop_in_bin
    
    return ece

def evaluate_model(model, model_name: str, X_test: List[str], y_test: np.ndarray) -> Dict:
    """Evaluate a trained model on test data."""
    print(f"\n📊 Evaluating {model_name.upper()} model...")
    
    # Measure latency
    latencies = []
    for text in X_test[:100]:  # Sample for latency measurement
        start = time.perf_counter()
        model.predict_proba([text])
        latencies.append(time.perf_counter() - start)
    
    # Get predictions
    start = time.perf_counter()
    y_proba = model.predict_proba(X_test)
    y_pred = model.predict(X_test)
    inference_time = time.perf_counter() - start
    
    metrics = compute_metrics(y_test, y_pred, y_proba, latencies)
    
    print(f"✅ {model_name.upper()} evaluation completed!")
    print(f"   Accuracy: {metrics['accuracy']:.4f}")
    print(f"   F1 Score: {metrics['f1']:.4f}")
    print(f"   ROC-AUC: {metrics['roc_auc']:.4f}")
    print(f"   ECE: {metrics['ece']:.4f}")
    if metrics['avg_latency_ms']:
        print(f"   Avg Latency: {metrics['avg_latency_ms']:.2f}ms")
    
    return metrics

print("✅ Evaluation functions defined!")


✅ Evaluation functions defined!


## Configuration and Data Loading


In [10]:
# Configuration
CONFIG = {
    'data_path': '/indonesian_hate_speech.csv',
    'output_dir': Path('/'),
    'random_seed': 42,
    'test_size': 0.2,
    'val_size': 0.1,
    
    # Model-specific configurations
    'tfidf': {
        'max_features': 10000,
        'ngram_range': (1, 2),
        'min_df': 2,
        'max_df': 0.95,
        'C': 1.0,
    },
    
    'bilstm': {
        'vocab_size': 10000,
        'embedding_dim': 128,
        'hidden_dim': 256,
        'num_layers': 2,
        'dropout': 0.3,
        'max_length': 200,
        'epochs': 10,
        'batch_size': 32,
        'learning_rate': 0.001,
    },
    
    'transformer': {
        'model_name': 'indobenchmark/indobert-base-p1',
        'max_length': 256,
        'epochs': 2,
        'batch_size': 16,
        'learning_rate': 2e-5,
        'warmup_steps': 50,
    }
}

# Set random seed for reproducibility
np.random.seed(CONFIG['random_seed'])
torch.manual_seed(CONFIG['random_seed'])

# Create output directory
CONFIG['output_dir'].mkdir(exist_ok=True)

print("🔧 Configuration loaded:")
print(f"   Data path: {CONFIG['data_path']}")
print(f"   Output directory: {CONFIG['output_dir']}")
print(f"   Random seed: {CONFIG['random_seed']}")
print(f"   GPU available: {torch.cuda.is_available()}")


🔧 Configuration loaded:
   Data path: /indonesian_hate_speech.csv
   Output directory: /
   Random seed: 42
   GPU available: True


In [11]:
# Load and prepare dataset
print("📊 Loading dataset...")

# Load the dataset
df = pd.read_csv(CONFIG['data_path'])
print(f"Dataset loaded: {len(df)} samples")

# Clean the data
df = df.dropna(subset=["text", "labels"])
df = df[df["text"].str.strip().str.len() > 0]

print(f"After cleaning: {len(df)} samples")
print(f"Toxicity rate: {df['labels'].mean():.2%}")

# Split the data
stratify_col = df["labels"] if CONFIG['test_size'] > 0 else None
train_val_df, test_df = train_test_split(
    df, test_size=CONFIG['test_size'], random_state=CONFIG['random_seed'], stratify=stratify_col
)

stratify_col = train_val_df["labels"] if CONFIG['val_size'] > 0 else None
train_df, val_df = train_test_split(
    train_val_df, test_size=CONFIG['val_size'], random_state=CONFIG['random_seed'], stratify=stratify_col
)

# Prepare training data
X_train = train_df["text"].tolist()
y_train = train_df["labels"].values
X_val = val_df["text"].tolist()
y_val = val_df["labels"].values
X_test = test_df["text"].tolist()
y_test = test_df["labels"].values

print(f"\n📋 Data split:")
print(f"   Training: {len(X_train):,} samples")
print(f"   Validation: {len(X_val):,} samples")
print(f"   Test: {len(X_test):,} samples")

# Show distribution
print(f"\n📈 Label distribution:")
print(f"   Train - Non-toxic: {np.sum(y_train == 0):,}, Toxic: {np.sum(y_train == 1):,}")
print(f"   Val   - Non-toxic: {np.sum(y_val == 0):,}, Toxic: {np.sum(y_val == 1):,}")
print(f"   Test  - Non-toxic: {np.sum(y_test == 0):,}, Toxic: {np.sum(y_test == 1):,}")


📊 Loading dataset...
Dataset loaded: 14306 samples
After cleaning: 14306 samples
Toxicity rate: 42.29%

📋 Data split:
   Training: 10,299 samples
   Validation: 1,145 samples
   Test: 2,862 samples

📈 Label distribution:
   Train - Non-toxic: 5,943, Toxic: 4,356
   Val   - Non-toxic: 661, Toxic: 484
   Test  - Non-toxic: 1,652, Toxic: 1,210


## Model Training


### Tier 1: TF-IDF + Logistic Regression


In [13]:
# Train TF-IDF model
print("🚀 Starting TF-IDF model training...")
tfidf_model = TFIDFModel(**CONFIG['tfidf'])

# Train the model
tfidf_metrics = tfidf_model.train(X_train, y_train, X_val, y_val)

# Save the model
tfidf_model.save(CONFIG['output_dir'] / 'tfidf')
print(f"💾 TF-IDF model saved to: {CONFIG['output_dir'] / 'tfidf'}")

# Evaluate on test set
tfidf_test_metrics = evaluate_model(tfidf_model, 'tfidf', X_test, y_test)

# Show feature importance
print("\n🔍 Top 10 most important features:")
feature_importance = tfidf_model.get_feature_importance(top_k=10)
for feature, importance in feature_importance.items():
    direction = "toxic" if importance > 0 else "non-toxic"
    print(f"   {feature}: {importance:.4f} ({direction})")


🚀 Starting TF-IDF model training...
🚀 Training TFIDF_LR model...
✅ TFIDF_LR training completed! F1: 0.9100
💾 TF-IDF model saved to: /tfidf

📊 Evaluating TFIDF model...
✅ TFIDF evaluation completed!
   Accuracy: 0.8326
   F1 Score: 0.8332
   ROC-AUC: 0.9173
   ECE: 0.0290
   Avg Latency: 0.86ms

🔍 Top 10 most important features:
   cebong: 6.2570 (toxic)
   2019gantipresiden: 6.1183 (toxic)
   lu: 4.9736 (toxic)
   jokowi: 4.6905 (toxic)
   tolol: 4.2435 (toxic)
   prabowo: 4.1731 (toxic)
   bubarkan: 4.0533 (toxic)
   lengserkan: 4.0349 (toxic)
   aku: -3.8675 (non-toxic)
   dasar: 3.8461 (toxic)


### Tier 2: BiLSTM


In [14]:
# Train BiLSTM model
print("🚀 Starting BiLSTM model training...")
bilstm_model = BiLSTMModel(**CONFIG['bilstm'])

# Train the model
bilstm_metrics = bilstm_model.train(X_train, y_train, X_val, y_val)

# Save the model
bilstm_model.save(CONFIG['output_dir'] / 'bilstm')
print(f"💾 BiLSTM model saved to: {CONFIG['output_dir'] / 'bilstm'}")

# Evaluate on test set
bilstm_test_metrics = evaluate_model(bilstm_model, 'bilstm', X_test, y_test)


🚀 Starting BiLSTM model training...
🚀 Training BILSTM model...
Epoch 1/10, Loss: 0.6824
Epoch 6/10, Loss: 0.6815
✅ BILSTM training completed! F1: 0.4223
💾 BiLSTM model saved to: /bilstm

📊 Evaluating BILSTM model...
✅ BILSTM evaluation completed!
   Accuracy: 0.5772
   F1 Score: 0.4225
   ROC-AUC: 0.5000
   ECE: 0.0034
   Avg Latency: 5.46ms


### Tier 3: IndoBERT Transformer


In [29]:
def clear_gpu_memory():
    """Clear GPU memory and run garbage collection."""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    gc.collect()
    print("🧹 GPU memory cleared")

clear_gpu_memory()

🧹 GPU memory cleared


In [12]:
def get_gpu_memory_info():
    """Get current GPU memory usage."""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3  # GB
        reserved = torch.cuda.memory_reserved() / 1024**3    # GB
        print(f"🎮 GPU Memory - Allocated: {allocated:.2f}GB, Reserved: {reserved:.2f}GB")
    else:
        print("🎮 No GPU available")
get_gpu_memory_info()

🎮 GPU Memory - Allocated: 0.00GB, Reserved: 0.00GB


In [ ]:
# Train Transformer model
print("🚀 Starting IndoBERT model training...")
transformer_model = TransformerModel(**CONFIG['transformer'])

# Train the model
transformer_metrics = transformer_model.train(X_train, y_train, X_val, y_val)

# Save the model
transformer_model.save(CONFIG['output_dir'] / 'transformer')
print(f"💾 Transformer model saved to: {CONFIG['output_dir'] / 'transformer'}")

# Evaluate on test set
transformer_test_metrics = evaluate_model(transformer_model, 'transformer', X_test, y_test)

# Show attention weights for a sample
print("\n🔍 Attention weights analysis:")
sample_texts = [
    "Kamu sangat bodoh dan tidak berguna",  # Toxic
    "Terima kasih atas bantuan yang luar biasa",  # Non-toxic
]

for text in sample_texts:
    print(f"\n📝 Text: '{text}'")
    try:
        tokens, attention = transformer_model.get_attention_weights(text)
        top_indices = np.argsort(attention.mean(axis=0))[-5:][::-1]
        print("   Top attended tokens:")
        for idx in top_indices:
            if idx < len(tokens):
                print(f"     {tokens[idx]}: {attention.mean(axis=0)[idx]:.4f}")
    except Exception as e:
        print(f"   Error getting attention: {e}")


🚀 Starting IndoBERT model training...
🚀 Training INDOBERT model...
Loading IndoBERT tokenizer and model: indobenchmark/indobert-base-p1
Epoch 1/2, Loss: 0.3679


## Results and Model Comparison


In [ ]:
# Compare all models
print("📊 Model Comparison Results:")
print("=" * 80)

# Create comparison dataframe
comparison_data = {
    'Model': ['TF-IDF', 'BiLSTM', 'IndoBERT'],
    'Tier': ['Basic', 'Contextual', 'Sociolinguistic'],
    'Accuracy': [tfidf_test_metrics['accuracy'], bilstm_test_metrics['accuracy'], transformer_test_metrics['accuracy']],
    'F1 Score': [tfidf_test_metrics['f1'], bilstm_test_metrics['f1'], transformer_test_metrics['f1']],
    'ROC-AUC': [tfidf_test_metrics['roc_auc'], bilstm_test_metrics['roc_auc'], transformer_test_metrics['roc_auc']],
    'PR-AUC': [tfidf_test_metrics['pr_auc'], bilstm_test_metrics['pr_auc'], transformer_test_metrics['pr_auc']],
    'ECE': [tfidf_test_metrics['ece'], bilstm_test_metrics['ece'], transformer_test_metrics['ece']],
    'Latency (ms)': [
        tfidf_test_metrics['avg_latency_ms'] or 0,
        bilstm_test_metrics['avg_latency_ms'] or 0,
        transformer_test_metrics['avg_latency_ms'] or 0
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False, float_format='%.4f'))

# Find best model by F1 score
best_f1_idx = comparison_df['F1 Score'].idxmax()
best_model = comparison_df.iloc[best_f1_idx]
print(f"\n🏆 Best model by F1 Score: {best_model['Model']} ({best_model['F1 Score']:.4f})")

# Find fastest model
fastest_idx = comparison_df['Latency (ms)'].idxmin()
fastest_model = comparison_df.iloc[fastest_idx]
print(f"⚡ Fastest model: {fastest_model['Model']} ({fastest_model['Latency (ms)']:.2f}ms)")

# Find best calibrated model (lowest ECE)
best_cal_idx = comparison_df['ECE'].idxmin()
best_cal_model = comparison_df.iloc[best_cal_idx]
print(f"🎯 Best calibrated model: {best_cal_model['Model']} (ECE: {best_cal_model['ECE']:.4f})")


In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Metrics to plot
metrics = ['Accuracy', 'F1 Score', 'ROC-AUC', 'PR-AUC', 'ECE', 'Latency (ms)']
colors = ['skyblue', 'lightcoral', 'lightgreen']

for i, metric in enumerate(metrics):
    row = i // 3
    col = i % 3
    
    axes[row, col].bar(comparison_df['Model'], comparison_df[metric], 
                      color=colors, alpha=0.7, edgecolor='black')
    axes[row, col].set_title(f'{metric} Comparison')
    axes[row, col].set_ylabel(metric)
    axes[row, col].tick_params(axis='x', rotation=45)
    
    # Add value labels on bars
    for j, v in enumerate(comparison_df[metric]):
        axes[row, col].text(j, v + 0.01, f'{v:.3f}', 
                           ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

# Performance vs Latency scatter plot
plt.figure(figsize=(10, 6))
plt.scatter(comparison_df['Latency (ms)'], comparison_df['F1 Score'], 
           s=200, c=colors, alpha=0.7, edgecolors='black')

for i, model in enumerate(comparison_df['Model']):
    plt.annotate(model, 
                (comparison_df['Latency (ms)'].iloc[i], comparison_df['F1 Score'].iloc[i]),
                xytext=(5, 5), textcoords='offset points', fontsize=12, fontweight='bold')

plt.xlabel('Latency (ms)')
plt.ylabel('F1 Score')
plt.title('Performance vs Latency Trade-off')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Generate detailed classification reports
models = {
    'TF-IDF': tfidf_model,
    'BiLSTM': bilstm_model,
    'IndoBERT': transformer_model
}

for model_name, model in models.items():
    print(f"\n📋 Detailed Classification Report - {model_name}")
    print("=" * 60)
    
    # Get predictions
    y_pred = model.predict(X_test)
    
    # Classification report
    report = classification_report(y_test, y_pred, 
                                  target_names=['Non-toxic', 'Toxic'],
                                  digits=4)
    print(report)
    
    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Non-toxic', 'Toxic'],
                yticklabels=['Non-toxic', 'Toxic'])
    plt.title(f'Confusion Matrix - {model_name}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.show()


## Sample Predictions and Final Summary


In [ ]:
# Test predictions with sample texts
test_texts = [
    "Terima kasih atas bantuan yang luar biasa",  # Non-toxic
    "Kamu sangat bodoh dan tidak berguna",      # Toxic
    "Saya tidak setuju dengan pendapat Anda",   # Non-toxic
    "Dasar goblok, mati aja lu",               # Toxic
    "Ini adalah hari yang indah",              # Non-toxic
]

print("🧪 Sample Predictions:")
print("=" * 80)

for text in test_texts:
    print(f"\n📝 Text: '{text}'")
    print("-" * 50)
    
    for model_name, model in models.items():
        proba = model.predict_proba([text])[0]
        prediction = "TOXIC" if proba > 0.5 else "NON-TOXIC"
        confidence = proba if proba > 0.5 else (1 - proba)
        
        print(f"   {model_name:8}: {prediction:10} (confidence: {confidence:.2%})")

print("\n✅ Sample predictions completed!")


In [ ]:
# Final Summary and Recommendations
print("🎯 Training Summary and Recommendations")
print("=" * 50)

print("\n📈 Performance Summary:")
for _, row in comparison_df.iterrows():
    print(f"\n{row['Model']} ({row['Tier']} Tier):")
    print(f"   • F1 Score: {row['F1 Score']:.4f}")
    print(f"   • ROC-AUC: {row['ROC-AUC']:.4f}")
    print(f"   • Latency: {row['Latency (ms)']:.2f}ms")
    print(f"   • Calibration (ECE): {row['ECE']:.4f}")

print("\n🏆 Model Recommendations:")
print(f"\n1. 🚀 For High-Throughput Applications:")
print(f"   Use {fastest_model['Model']} - {fastest_model['Latency (ms)']:.2f}ms latency")
print(f"   Best for: Real-time filtering, high-volume processing")

print(f"\n2. 🎯 For Best Accuracy:")
print(f"   Use {best_model['Model']} - {best_model['F1 Score']:.4f} F1 score")
print(f"   Best for: Critical applications where accuracy is paramount")

print(f"\n3. ⚖️ For Balanced Performance:")
print(f"   Consider BiLSTM - Good balance of accuracy and speed")
print(f"   Best for: General-purpose toxicity detection")

print("\n💡 Usage Guidelines:")
print("   • Tier 1 (TF-IDF): Use for initial filtering and high-volume scenarios")
print("   • Tier 2 (BiLSTM): Use for balanced performance requirements")
print("   • Tier 3 (IndoBERT): Use for highest accuracy needs")

print("\n🔧 Next Steps:")
print("   1. Download trained models from Kaggle output")
print("   2. Deploy models using the API endpoints")
print("   3. Monitor performance in production")
print("   4. Retrain periodically with new data")

print("\n✅ All models trained and saved successfully!")
print(f"   Models saved to: {CONFIG['output_dir']}")
print("   Ready for deployment via CLI or API")

# Show final model files
print(f"\n📁 Saved model files:")
for model_dir in CONFIG['output_dir'].iterdir():
    if model_dir.is_dir():
        print(f"   {model_dir.name}/")
        for file in model_dir.iterdir():
            print(f"     - {file.name}")
